# Initial Exploration
* Tensorboard visualization
* Checking weights of other nodes don't change when downstream node is learned

In [1]:
import numpy as np
import tensorflow as tf
%load_ext tensorboard

In [2]:
class Node(tf.Module):
    def __init__(self, name=None):
        super(Node, self).__init__(name=name)
        with self.name_scope:
            self.dense_1 = tf.keras.layers.Dense(
                32, 
                activation='relu'
            )
            self.dense_2 = tf.keras.layers.Dense(
                1, 
                activation='linear'
            )
        
    @tf.Module.with_name_scope
    def __call__(self, x):
        x = self.dense_1(x)
        x = self.dense_2(x)
        return x

In [3]:
!rm -rf logs/

In [4]:
@tf.function
def compute(x, layer1, layer2):
    all_outputs = []
    
    layer_outputs = []
    with tf.name_scope('Layer1') as scope:
        for n in layer1:
            x = tf.identity(
                n(x), n.name + '_output' 
            )
            layer_outputs.append(x)
        
    x = tf.concat(layer_outputs, axis=-1)
    all_outputs.append(x)
    layer_outputs = []
    with tf.name_scope('Layer2') as scope:
        for n in layer2:
            x = tf.identity(
                n(x), n.name + '_output' 
            )
            layer_outputs.append(x)
            
    x = tf.concat(layer_outputs, axis=-1)
    all_outputs.append(x)
    return all_outputs

l1_n1 = Node(name='l1_n1')
l1_n2 = Node(name='l1_n2')
l1_n3 = Node(name='l1_n3')

l2_n1 = Node(name='l2_n1')
l2_n2 = Node(name='l2_n2')

x = tf.random.uniform((1, 3))
compute(
    x, 
    layer1=[l1_n1, l1_n2, l1_n3],
    layer2=[l2_n1, l2_n2],
)

# Set up logging.
logdir = 'logs/layer/' 
writer = tf.summary.create_file_writer(logdir)

tf.summary.trace_on(graph=True, profiler=True)

# Call only one tf.function when tracing.
outputs = compute(
    x, 
    layer1=[l1_n1, l1_n2, l1_n3],
    layer2=[l2_n1, l2_n2],
)

with writer.as_default():
    tf.summary.trace_export(
        name="my_func_trace",
        step=0,
        profiler_outdir=logdir
    )

Instructions for updating:
use `tf.profiler.experimental.start` instead.
Instructions for updating:
use `tf.profiler.experimental.stop` instead.
Instructions for updating:
`tf.python.eager.profiler` has deprecated, use `tf.profiler` instead.
Instructions for updating:
`tf.python.eager.profiler` has deprecated, use `tf.profiler` instead.


In [5]:
%tensorboard --logdir logs/layer

Reusing TensorBoard on port 6006 (pid 9237), started 1 day, 19:45:08 ago. (Use '!kill 9237' to kill it.)

In [11]:
opt1 = tf.keras.optimizers.SGD(learning_rate=0.01)
opt2 = tf.keras.optimizers.SGD(learning_rate=0.01)
y_true = np.array([[1]])
x = tf.random.uniform((1, 3)) 

with tf.GradientTape(persistent=True) as tape:    
    outputs = compute(
        x, 
        layer1=[l1_n1, l1_n2, l1_n3],
        layer2=[l2_n1, l2_n2],
    )
        
def learn(one=True):
#     x = tf.random.uniform((1, 3))   
    if one:
        
        tape.__enter__()
        loss1 = (5 - outputs[0][0,0])**2
        tape.__exit__(None, None, None)
        
        grads = tape.gradient(loss1, l1_n1.trainable_variables)
        opt1.apply_gradients(
            zip(grads, l1_n1.variables)
        ) 
        print(loss1)
    else:
        
        tape.__enter__()
        loss2 = (10 - outputs[1][0,0])**2
        tape.__exit__(None, None, None)
        
        grads = tape.gradient(loss2, l2_n1.trainable_variables)
        opt2.apply_gradients(
            zip(grads, l2_n1.variables)
        ) 
        print(loss2)

# To Do
* Create multiple nodes 
    * this will become a layer
    * forward pass through all
    * concatenate ouputs
    * feed to next layer
    * track outputs 

In [12]:
l1_n1.dense_1.weights[0][0]

<tf.Tensor: shape=(32,), dtype=float32, numpy=
array([-0.16980888,  0.25254482, -0.06205559, -0.22984487,  0.34059486,
        0.02355254,  0.07406407,  0.2760114 ,  0.30084175, -0.12324053,
        0.01292482, -0.07519793,  0.29028597,  0.15742543,  0.14542884,
       -0.31154647,  0.03229836,  0.34002203,  0.39559618, -0.08305359,
       -0.28671616,  0.17510697,  0.27431428, -0.04398141,  0.08727428,
       -0.3376494 ,  0.04782134,  0.01094621, -0.18996671,  0.07321462,
        0.39087903, -0.1496613 ], dtype=float32)>

In [20]:
learn()
l1_n1.dense_1.weights[0][0]

tf.Tensor(21.106457, shape=(), dtype=float32)


<tf.Tensor: shape=(32,), dtype=float32, numpy=
array([-0.16980888,  0.09479891, -0.20142595, -0.22984487,  0.16036105,
       -0.1143382 ,  0.06706259,  0.3962234 ,  0.538343  , -0.12324053,
        0.01292482, -0.2896153 ,  0.48736575,  0.15742543,  0.20677245,
       -0.31154647, -0.19838639,  0.57012564,  0.37853116, -0.08305359,
       -0.28671616,  0.0617844 ,  0.4836075 , -0.22375345,  0.08946382,
       -0.3376494 ,  0.04782134,  0.01094621, -0.18996671,  0.17519876,
        0.60667765, -0.1496613 ], dtype=float32)>

In [37]:
l1_n1.dense_1.weights[0][0]

<tf.Tensor: shape=(32,), dtype=float32, numpy=
array([-0.16980888,  0.09479891, -0.20142595, -0.22984487,  0.16036105,
       -0.1143382 ,  0.06706259,  0.3962234 ,  0.538343  , -0.12324053,
        0.01292482, -0.2896153 ,  0.48736575,  0.15742543,  0.20677245,
       -0.31154647, -0.19838639,  0.57012564,  0.37853116, -0.08305359,
       -0.28671616,  0.0617844 ,  0.4836075 , -0.22375345,  0.08946382,
       -0.3376494 ,  0.04782134,  0.01094621, -0.18996671,  0.17519876,
        0.60667765, -0.1496613 ], dtype=float32)>

In [22]:
l2_n1.dense_1.weights[0][0]

<tf.Tensor: shape=(32,), dtype=float32, numpy=
array([ 3.2356516e-01, -3.8472632e-01,  8.2865238e-02, -2.7441099e-01,
        1.6629279e-02, -2.4246565e-01, -3.5992622e-01,  9.2643052e-02,
        2.0581484e-04,  1.4818373e-01,  3.0263606e-01, -3.4938240e-01,
       -1.5739471e-02, -3.2327881e-01,  1.9117448e-01, -3.7518239e-01,
       -3.7235126e-01, -1.2324318e-01,  1.4464691e-01,  3.9056638e-01,
        3.6470374e-01, -2.7473706e-01, -3.2218397e-02,  3.8885716e-01,
        3.2914510e-01,  3.6735177e-02,  4.0864590e-01, -2.6985055e-01,
        3.7271795e-01, -2.5871772e-01, -5.9182703e-02,  3.6960840e-02],
      dtype=float32)>

In [34]:
learn(False)
l2_n1.dense_1.weights[0][0]

tf.Tensor(102.313995, shape=(), dtype=float32)


<tf.Tensor: shape=(32,), dtype=float32, numpy=
array([-0.05255918, -0.38472632, -0.28360093, -0.274411  ,  0.00814488,
       -0.24246565, -0.35992622,  0.16549775, -0.36757907,  0.55983484,
        0.38509968, -0.3493824 , -0.01573947, -0.3232788 ,  0.58463955,
       -0.3751824 , -0.37235126, -0.12324318,  0.00380771,  0.30381623,
        0.4215301 , -0.27473706, -0.0322184 ,  0.11039568, -0.0300249 ,
        0.42510796,  0.42013004, -0.26985055,  0.23112738, -0.25871772,
       -0.0591827 , -0.05498439], dtype=float32)>

In [2]:
# # The function to be traced.
@tf.function
def my_func(l1, l2, x):
    o1 = tf.nn.relu(l1(x))
    _o1 = tf.stop_gradient(o1)

    o2 = tf.nn.relu(l2(o1))
    return o1, o2

# # Set up logging.
# logdir = 'logs/stop/' 
# writer = tf.summary.create_file_writer(logdir)

# # Sample data for your function.
# x = tf.random.uniform((3,3))
# y = tf.random.uniform((3,3))
# z = tf.random.uniform((3,3))

# # Bracket the function call with
# # tf.summary.trace_on() and tf.summary.trace_export().
# tf.summary.trace_on(graph=True, profiler=True)
# # Call only one tf.function when tracing.
# z = my_func(x, y, z)
# with writer.as_default():
#   tf.summary.trace_export(
#       name="my_func_trace",
#       step=0,
#       profiler_outdir=logdir)

Instructions for updating:
use `tf.profiler.experimental.start` instead.
Instructions for updating:
use `tf.profiler.experimental.stop` instead.
Instructions for updating:
`tf.python.eager.profiler` has deprecated, use `tf.profiler` instead.
Instructions for updating:
`tf.python.eager.profiler` has deprecated, use `tf.profiler` instead.
